# Aula 03 · Vídeo 2 — Detecção e Anonimização de PII com Microsoft Presidio

> **Disciplina:** Segurança, Guardrails e Conformidade para LLMs · **Caso condutor:** Banco **Aurora** / **Aurora Assist**
> **Tipo:** conceitual **+** prático · **Doc 04:** seções 3-8 (PII em LLMs, k-anon/DP, Presidio, recognizers, tokenização, embeddings)

## 🎯 Objetivos
- Detectar **PII brasileira** (CPF/CNPJ/RG/CEP/conta) combinando *regex* + contexto + **dígito verificador**.
- Medir o equilíbrio **falso negativo × falso positivo** (recall/precision/F1 e **taxa de vazamento**).
- Aplicar **anonimização reversível** (o LLM vê só tokens) e **minimização** de dados.
- Entender por que é preciso **anonimizar ANTES de gerar embeddings**.

> [!WARNING]
> ⚠️ **Uso ético.** Todos os dados são **sintéticos**; CPF/CNPJ têm apenas dígitos verificadores válidos, sem corresponder a pessoas reais.

## 1. O problema da PII em LLMs (Doc 04, §3)

Em sistemas tradicionais a PII vive em bancos estruturados, com controle de acesso direto. Em LLMs, ela escapa para **quatro lugares perigosos**: os **prompts**, as **respostas**, os **logs** de observabilidade e — o mais sutil — os **embeddings**. Some-se a **memorização involuntária** (Carlini et al., 2021): modelos podem regurgitar PII do treino. É o **LLM06** (*Sensitive Information Disclosure*) do OWASP, agora visto pela ótica do dado. Para o Aurora, deixar o CPF circular não é só risco técnico — é incidente de segurança e violação de LGPD.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Offline-first e determinístico (Restart & Run All sempre funciona).
os.environ.setdefault('PRESIDIO_MODE', 'deterministic')
os.environ.setdefault('TOKEN_VAULT_BACKEND', 'memory')

import random
random.seed(42)
print('ROOT =', ROOT)

In [ ]:
# Garante os datasets sintéticos (seed fixa). Regenera se ausente.
p = ROOT / 'data' / 'pii_samples' / 'pii_samples.jsonl'
if not p.exists():
    import subprocess
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_datasets.py')], check=True)
print('dataset pronto:', p.exists())

## 2. Presidio: Analyzer + Anonymizer (Doc 04, §5)

O **Microsoft Presidio** tem dois componentes: o **Analyzer** detecta entidades combinando *regex*, **NER** e **checksum**; o **Anonymizer** mascara/substitui/remove conforme a política. Aqui usamos a *engine* **determinística e offline** do repositório (mesma lógica de detecção), que é o *fallback* do Presidio e roda sem baixar modelos. No Lab (vídeo 4) trocamos por Presidio real.

In [ ]:
from src.security import get_analyzer

analyzer = get_analyzer('deterministic', 'pt')
texto = 'Cliente Ana Souza, CPF 529.982.247-25, e-mail ana@example.com, conta corrente 45678-9.'
for m in analyzer.analyze(texto, min_score=0.5):
    print(f'{m.entity.value:16} score={m.score:.2f}  ->  {m.text}')

## 3. Recognizer customizado + checksum (Doc 04, §6 / Snippet 1)

O Presidio não conhece documentos brasileiros por padrão. O **Snippet 1** do Doc 04 cria um *recognizer* de CPF combinando *regex* + `context` + **validação de dígito verificador**. A validação é decisiva: sem ela, qualquer sequência de 11 dígitos viraria CPF (falso positivo); com ela, só números matematicamente válidos passam — reduzindo FP **sem** aumentar FN.

In [ ]:
from src.security import is_valid_cpf, detect_pii

print('529.982.247-25 valido? ', is_valid_cpf('529.982.247-25'))
print('111.111.111-11 valido? ', is_valid_cpf('111.111.111-11'))

# Sem checksum, o CPF invalido seria um FALSO POSITIVO; a engine o descarta:
print('deteccoes em texto com CPF invalido:', detect_pii('numero 111.111.111-11 aqui', min_score=0.5))

## 4. Fundamentos: k-anonimato, l-diversidade e Differential Privacy (Doc 04, §4)

A anonimização robusta tem base matemática. O **k-anonimato** (Sweeney, 2002) exige que cada registro seja indistinguível de ao menos k-1 outros; a **l-diversidade** (Machanavajjhala et al., 2007) pede variedade no atributo sensível. O ápice é a **Differential Privacy** (Dwork, 2006), com garantia formal: para D1 e D2 diferindo em um indivíduo,  Pr[M(D1) ∈ S] ≤ e^ε · Pr[M(D2) ∈ S]. Quanto menor o **ε**, mais forte a privacidade (e menor a utilidade). Essas ideias guiam *quanto* podemos generalizar/ruidar dados agregados — enquanto valores transacionais individuais devem ser **tokenizados** e mantidos fora do alcance do modelo.

## 5. Avaliação da detecção: recall, precision, F1 e taxa de vazamento (Doc 04, §6)

Um detector que erra muito é tão perigoso quanto a ausência de detector. Medimos sobre o corpus sintético `data/pii_samples` — que inclui amostras **benignas** (sem PII) como baseline de FP.

In [ ]:
import json
from src.security import detection_confusion

rows = [json.loads(l) for l in (ROOT / 'data' / 'pii_samples' / 'pii_samples.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
expected = [r.get('entities', []) for r in rows]
predicted = [[m.entity.value for m in analyzer.analyze(r['text'], min_score=0.5)] for r in rows]

cm = detection_confusion(expected, predicted)
print('amostras:', len(rows))
print('metricas:', cm.to_dict())
print('taxa de vazamento (leak_rate):', cm.leak_rate)

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

counts = Counter(e for preds in predicted for e in preds)
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(list(counts.keys()), list(counts.values()), color='#3b7dd8')
ax.set_title('PII detectada por entidade (data/pii_samples)')
ax.set_ylabel('ocorrencias')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Anonimização reversível com cofre de tokens (Doc 04, §7 / Snippet 2)

Anonimizar não é destruir. Cada PII vira um **token determinístico** (o mesmo valor gera sempre o mesmo token) e o mapa token↔valor fica num **cofre** (Azure Key Vault em produção). O LLM processa apenas o token; só um serviço autorizado, **fora** do alcance do modelo, reverte quando necessário.

In [ ]:
from src.security import ReversibleAnonymizer, DeterministicAnalyzer, InMemoryTokenVault

anon = ReversibleAnonymizer(analyzer=DeterministicAnalyzer('pt'), vault=InMemoryTokenVault(secret='lab'), min_score=0.5)
res = anon.anonymize('Meu CPF e 529.982.247-25 e a conta corrente 45678-9.', allowed_pii=['BR_CPF', 'BR_ACCOUNT'])
print('LLM ve      :', res.text)
print('tokens rev. :', res.reversible_tokens)
print('reidratado  :', anon.deanonymize(res.text))

## 7. Minimização de dados (Doc 04, §12)

*O dado mais seguro é aquele que não existe.* Se o fluxo não precisa de uma entidade, ela é **removida já na entrada** — reduzindo superfície de ataque e impacto de vazamento.

In [ ]:
faq = anon.anonymize('Meu CPF e 529.982.247-25.', allowed_pii=[])
print('faq (minimizado):', faq.text)
print('removido por minimizacao:', [m.entity.value for m in faq.minimized])

## 8. Vazamento por embeddings (Doc 04, §8)

Embeddings **não são anônimos**: há ataques de **inversão** (Song & Raghunathan, 2020) e de **inferência de pertinência** (Shokri et al., 2017). Armazenar embeddings de documentos com CPF é, na prática, armazenar dado pessoal. A mitigação é **anonimizar ANTES de vetorizar**.

In [ ]:
doc = 'Protocolo: cliente Ana Souza, CPF 529.982.247-25, email ana@example.com.'
to_index = anon.anonymize(doc, allowed_pii=[]).text
print('texto a vetorizar:', to_index)
print('PII residual     :', detect_pii(to_index, min_score=0.5))

## ✅ Resumo e gancho

Detectamos PII BR com *regex* + contexto + checksum, medimos FN/FP e aplicamos **anonimização reversível** e **minimização**, protegendo inclusive os embeddings. No **vídeo 3** blindamos o outro vetor de ataque: as **ferramentas** que o modelo aciona — com MCP isolado e *capability tokens*.